In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
def increment_by_one(number):
    return number + 1   

In [4]:
from langchain_core.runnables import RunnableLambda

increment_runnable = RunnableLambda(increment_by_one) # this is how we create a runnable from a function

incremented_value = increment_runnable.invoke(5)
print(f"Incremented Value: {incremented_value}")

Incremented Value: 6


In [5]:
from langchain_core.runnables import RunnableSequence

sequence = RunnableLambda(lambda x: x + 1) | RunnableLambda(lambda x: x * 2)

print(sequence.invoke(5))

12


In [6]:
from langchain_core.runnables import RunnableParallel
# dictonary is passed in runnable parallel

rp = RunnableParallel(
    {
        'mul_2':RunnableLambda(lambda x:x*2),
        'mul_5':RunnableLambda(lambda x:x*5)
    }
)

rp.invoke(5)

{'mul_2': 10, 'mul_5': 25}

In [7]:
sequence = RunnableLambda(lambda x:x+1) | RunnableParallel(
    {
        'mul_2':RunnableLambda(lambda x:x*2),
        'mul_5':RunnableLambda(lambda x:x*5)
    }
)

sequence.invoke(5)

{'mul_2': 12, 'mul_5': 30}

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# PromptTemplate Vs ChatPromptTemplate 
# ✅ PromptTemplate
# Used for single text prompts (good for non-chat models or simple messages).
# ✅ ChatPromptTemplate
# Used for multi-message chat-style prompts (system / user / assistant roles).

text = f"""Needed a nice lamp for my bedroom, and this one had additional storage and not too high of a price point. Got it fast. The string to our lamp broke during the transit and the company happily sent over a new one. Came within a few days as well. It was easy to put together. I had a missing part, so I contacted their support and they very quickly got me the missing piece! Lumina seems to me to be a great company that cares about their customers and products!!"""

summarization_prompt = "summarize the below text into one sentence: {text}"

sentiment_prompt = "What is the sentiment of the text below : {text}"

fprompt = PromptTemplate(template=summarization_prompt, input_variables=["text"])
sprompt = PromptTemplate(template=sentiment_prompt, input_variables=["text"])

summary_chain = fprompt | llm | StrOutputParser()
sentiment_chain = sprompt | llm | StrOutputParser()

final_chain = RunnableParallel(
    {
        'summary':summary_chain,
        'sentiment':sentiment_chain
    }
)


final_chain.invoke({'text':text})


{'summary': 'Despite receiving a new lamp with a broken string and a missing part, the customer highly praises Lumina for their prompt, happy, and caring customer service in resolving the issues.',
 'sentiment': 'The sentiment of the text is **Strongly Positive**.\n\nHere\'s why:\n\n*   **Initial Positive Impressions:** The user liked the lamp\'s features ("nice lamp," "additional storage," "not too high of a price point") and speed of delivery ("Got it fast").\n*   **Excellent Problem Resolution:** Although there were two issues (a broken string and a missing part), the company\'s response to both was exemplary. They "happily sent over a new one" quickly and "very quickly got me the missing piece!"\n*   **Explicit Positive Conclusion:** The final sentence unequivocally states, "Lumina seems to me to be a great company that cares about their customers and products!!" This strongly positive summary overrides any minor initial inconveniences.'}

In [9]:
# runnable passthrough
from langchain_core.runnables import RunnablePassthrough

rps = RunnablePassthrough()

rps.invoke(
    {
        "language":"Python",
        "framework":"LangChain",
        "task":"Building LLM apps"
    }
)

{'language': 'Python', 'framework': 'LangChain', 'task': 'Building LLM apps'}

In [10]:
# we use runnable passthrough when we want to pass the input as it is to the next runnable in the sequence

rp = RunnableParallel(
    {
        "original_input":RunnablePassthrough(),
        "incremented_output":RunnableLambda(lambda x:x+1)
    }
)

rp.invoke(5)

{'original_input': 5, 'incremented_output': 6}

In [11]:
from langchain_core.runnables import RunnablePick

r = RunnablePick("language")

r.invoke({"language": "java", "task": "return a sum of numbers in a list. Do proper formatting of result"})

'java'

In [12]:
from langchain_core.runnables.passthrough import RunnableAssign
# this will assign key value pair for the given input

mapper = {
    "add_ten": RunnableLambda(lambda x: x["input"] + 10)
}

runnable_assign = RunnableAssign(mapper)

output = runnable_assign.invoke({"input": 5})
output

{'input': 5, 'add_ten': 15}